# EHR Copilot — ORPO v4 Training on Colab

**Goal:** Fine-tune Qwen2.5-Coder-7B-Instruct with ORPO to improve SQL generation and abstention on the EHRSQL MIMIC-III benchmark.

**Target:** RS(10) ≥ 0.813 (EHRSQL 2024 leaderboard top score — LG AI / KAIST)

**Recommended runtime:** A100 40GB (Runtime → Change runtime type → A100)

---

## Before you start

Upload `EHRCopilot_colab_data.zip` (347 MB) to your Google Drive at:

```
MyDrive/EHRCopilot/EHRCopilot_colab_data.zip
```

That zip contains everything: SQLite DB, EHRSQL data, ORPO pairs, embeddings, template classifier, ORPO v3 adapter, and source package.

---

## Steps
1. ▶ **Cell 1** — check GPU
2. ▶ **Cell 2** — install dependencies (~3 min)
3. ▶ **Cell 3** — mount Drive + extract data zip
4. ▶ **Cell 4** — verify all files present
5. ▶ **Cell 5** — detect GPU and set batch config
6. ▶ **Cell 6** — run ORPO v4 training (6–8 hrs A100, 24–36 hrs T4)
7. ▶ **Cell 7** — save adapter to Google Drive
8. ▶ **Cell 8** *(optional)* — RAGAS retrieval eval (confirms 92.7% recall+precision)
9. ▶ **Cell 9** *(optional)* — full eval + RS(10) score

In [ ]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────────────
import subprocess, torch

r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                    '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', r.stdout.strip())
print(f'PyTorch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: Install dependencies (~3 min) ─────────────────────────────────
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q \
    "trl>=0.9.0" "peft>=0.11.0" "bitsandbytes>=0.43.0" \
    "accelerate>=0.30.0" "transformers>=4.45.0" \
    "datasets>=2.19.0" "huggingface-hub>=0.23.0" \
    "sentence-transformers>=2.7.0" "rank-bm25>=0.2.2" \
    "scikit-learn>=1.4.0" "joblib>=1.3.0"
print('All dependencies installed.')

In [ ]:
# ── Cell 3: Mount Drive + extract data zip ────────────────────────────────
# Expects: MyDrive/EHRCopilot/EHRCopilot_colab_data.zip
from google.colab import drive
import zipfile, os, shutil

drive.mount('/content/drive')

DRIVE_ZIP = '/content/drive/MyDrive/EHRCopilot/EHRCopilot_colab_data.zip'
os.chdir('/content')

if not os.path.exists(DRIVE_ZIP):
    raise FileNotFoundError(
        f'Not found: {DRIVE_ZIP}\n'
        'Upload EHRCopilot_colab_data.zip to MyDrive/EHRCopilot/ and rerun.'
    )

size_mb = os.path.getsize(DRIVE_ZIP) / 1024 / 1024
print(f'Found {DRIVE_ZIP}  ({size_mb:.0f} MB)')
print('Extracting ...')

with zipfile.ZipFile(DRIVE_ZIP) as zf:
    zf.extractall('/content')

print('Done.')

In [ ]:
# ── Cell 4: Verify all files ───────────────────────────────────────────────
import os, sys

sys.path.insert(0, '/content/src')
os.environ['PYTHONPATH'] = '/content/src'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

checks = [
    ('data/mimic_iv_demo.db',                                           'SQLite DB (required for eval)'),
    ('data/ehrsql/orpo_v4_pairs.jsonl',                                 'ORPO training pairs'),
    ('data/ehrsql/template_classifier.pkl',                              'Template classifier'),
    ('data/ehrsql/train_embeddings_bge_large.npy',                      'BGE-large embeddings'),
    ('data/ehrsql/ehrsql/mimic_iii/train.json',                         'EHRSQL train split'),
    ('data/ehrsql/ehrsql/mimic_iii/test.json',                          'EHRSQL test split'),
    ('checkpoints/orpo_v3/adapter_final/adapter_model.safetensors',     'ORPO v3 adapter weights'),
    ('src/ehrcopilot/finetune/abstention_dpo.py',                       'Training script'),
    ('src/ehrcopilot/eval/harness.py',                                   'Eval harness'),
]

print('File check:')
all_ok = True
for path, desc in checks:
    ok = os.path.exists(path)
    size = f'  ({os.path.getsize(path)/1024/1024:.0f} MB)' if ok else ''
    print(f'  {"OK" if ok else "MISSING"}  {path}{size}  — {desc}')
    if not ok:
        all_ok = False

print('\nAll files present — ready to train!' if all_ok else '\nSome files missing — check the zip contents.')

In [ ]:
# ── Cell 6: Detect GPU and set batch config ────────────────────────────────
import os, sys, torch

os.chdir('/content')
sys.path.insert(0, '/content/src')
os.environ['PYTHONPATH'] = '/content/src'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f'GPU: {gpu_name}  ({vram_gb:.1f} GB VRAM)')

if vram_gb >= 35:    # A100 40GB
    BATCH_SIZE, GRAD_ACCUM, MAX_LENGTH = 4, 4, 1536
elif vram_gb >= 20:  # A100 20GB / L4
    BATCH_SIZE, GRAD_ACCUM, MAX_LENGTH = 2, 8, 1536
else:                # T4 16GB
    BATCH_SIZE, GRAD_ACCUM, MAX_LENGTH = 1, 16, 1024

print(f'batch={BATCH_SIZE}  grad_accum={GRAD_ACCUM}  max_length={MAX_LENGTH}')
print(f'Effective batch size: {BATCH_SIZE * GRAD_ACCUM}  (target: 16)')

In [ ]:
# ── Cell 7: Run ORPO v4 training ───────────────────────────────────────────
#
# Starting from ORPO v3 adapter (SFT + v1-v3 preference trained).
# 5,218 pairs: 4,856 SQL quality + 362 abstention.
# Loss target: < 0.05  |  Accuracy target: > 75%  |  Margin: consistently positive
#
# A100 40GB: ~6-8 hrs  |  T4 16GB: ~24-36 hrs

import subprocess, sys

cmd = [
    sys.executable, '-m', 'ehrcopilot.finetune.abstention_dpo',
    '--pairs',      'data/ehrsql/orpo_v4_pairs.jsonl',
    '--adapter',    'checkpoints/orpo_v3/adapter_final',
    '--output',     'checkpoints/orpo_v4',
    '--epochs',     '1',
    '--lr',         '5e-6',
    '--max-length', str(MAX_LENGTH),
]

print('Command:', ' '.join(cmd))
print('-' * 60)

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    env={**os.environ, 'PYTHONPATH': '/content/src', 'TOKENIZERS_PARALLELISM': 'false'},
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f'\nExit code: {proc.returncode}')

In [ ]:
# ── Cell 8: Save adapter to Google Drive ───────────────────────────────────
import shutil, os

SRC = '/content/checkpoints/orpo_v4'
DST = '/content/drive/MyDrive/EHRCopilot/checkpoints/orpo_v4'

if os.path.exists(SRC):
    if os.path.exists(DST):
        shutil.rmtree(DST)
    shutil.copytree(SRC, DST)
    print(f'Adapter saved to Google Drive: {DST}')
    for f in os.listdir(f'{DST}/adapter_final'):
        print(f'  {f}')
else:
    print('No checkpoint at', SRC, '— did training complete?')

In [ ]:
# ── Cell 9 (optional): RAGAS retrieval eval ────────────────────────────────
# Confirms template retrieval quality: target Recall@2 >= 90%, Precision@2 >= 90%.
# Expected: ~92.7% on both (pre-validated locally).

import subprocess, sys

result = subprocess.run([
    sys.executable, '-m', 'ehrcopilot.eval.rag_eval',
    '--train',             'data/ehrsql/ehrsql/mimic_iii/train.json',
    '--test',              'data/ehrsql/ehrsql/mimic_iii/test.json',
    '--mode',              'template',
    '--embed-cache',       'data/ehrsql/train_embeddings_bge_large.npy',
    '--classifier-cache',  'data/ehrsql/template_classifier.pkl',
    '--output',            'rag_eval_results.json',
    '--k',                 '1,2,5',
], capture_output=True, text=True,
   env={**os.environ, 'PYTHONPATH': '/content/src'})

print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])

In [ ]:
# ── Cell 10 (optional): Full eval — RS(N) scoring ─────────────────────────
# SQL generation → execution-guided repair → RS(0)/RS(5)/RS(10) scoring.
# Uses template retrieval (92.7% precision) for 2-shot examples.
# A100: ~1.5-2 hrs  |  T4: ~4-6 hrs

import subprocess, sys, json, os

ADAPTER = 'checkpoints/orpo_v4/adapter_final'
OUTPUT  = 'orpo_v4_eval_results.json'

proc = subprocess.Popen([
    sys.executable, '-m', 'ehrcopilot.eval.harness',
    'data/ehrsql/ehrsql/mimic_iii/test.json',
    '--model',            ADAPTER,
    '--output',           OUTPUT,
    '--repair',
    '--few-shot',         'data/ehrsql/ehrsql/mimic_iii/train.json',
    '--retrieval-mode',   'template',
    '--classifier-cache', 'data/ehrsql/template_classifier.pkl',
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
   env={**os.environ, 'PYTHONPATH': '/content/src', 'TOKENIZERS_PARALLELISM': 'false'})

for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

if os.path.exists(OUTPUT):
    r = json.load(open(OUTPUT))
    print('\n' + '='*50)
    print(f'  EX:     {r.get("EX", 0):.4f}')
    print(f'  RS(0):  {r.get("RS(0)", 0):.4f}')
    print(f'  RS(10): {r.get("RS(10)", 0):.4f}   (target ≥ 0.813)')
    print('='*50)